# Exercise 2 — Image Classification (From Scratch)
### Animal Face Classifier

---

## The Problem

A wildlife monitoring platform needs to automatically tag camera trap images
by animal type. You have been given a dataset of high-quality animal face
photographs across three categories: **cat, dog, and wild**.

Build an image classifier from scratch — no pretrained models.
The network must learn everything from the pixel data alone.

---

## The Data

```python
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/andrewmvd/animal-faces")
```

Images are organized in folders by class label under a nested directory structure.
You will need to walk the directory yourself to build your dataset.

⚠️ **One thing worth knowing:** images vary in size.
Your model needs a fixed input size — you decide what that is.

---

## What You Need to Deliver

A working Colab notebook that contains:

1. **A trained CNN** built from scratch in PyTorch that classifies animal faces
2. **A visualization** — a grid of sample images from the dataset with their labels
3. **A training report** — loss and accuracy curves for train and validation
4. **A test accuracy score**
5. **A live inference demo** — given any image file, the model returns the predicted animal

Your CNN must be built using `nn.Module` with convolutional layers.
`torchvision.models` pretrained weights are not allowed here — that is Exercise 3.

---

In [1]:
!git clone https://github.com/Ibraheem-Al-hafith/classification_hub.git
%cd classification_hub

Cloning into 'classification_hub'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 36 (delta 16), reused 22 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 270.93 KiB | 2.18 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/classification_hub


In [12]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from tqdm import tqdm

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    print(f"Random seed set to: {seed}")

class config:
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size = 32 # This batch_size is not directly used by dataloader_configs
    num_workers = 2
    learning_rate = 0.001
    num_epochs = 2
    dataset_name = "andrewmvd/animal-faces"
    h, w = 384, 384
    scaler = "standard" # options: "standard" , "minmax" , "robust"
    dataloader_configs = {
        "batch_size": 32, # Reduced batch size to mitigate OutOfMemoryError
        "num_workers": 2,
        "pin_memory": True
    }

seed_everything(config.seed)
print(f"Device set to: {config.device}")

Random seed set to: 42
Device set to: cuda


In [3]:
from utilities import download_kaggle_dataset
images_path = download_kaggle_dataset(config.dataset_name)

Using Colab cache for faster access to the 'animal-faces' dataset.
Path to the dataset: /kaggle/input/animal-faces
Is the path a file? False
Files inside the path: ['afhq']


In [4]:
"""Dataset utility module for processing image classification directory layouts.

This module provides a robust PyTorch Dataset capable of parsing structured image
folders, handling corrupted files gracefully during training, and visualizing
batches of samples.
"""

import logging
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple, Union

import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from torchvision.io import ImageReadMode, read_image
from torchvision.transforms import v2


@dataclass
class DatasetConfig:
    """Mock configuration matching your project structure for height and width."""
    h: int = 224
    w: int = 224


class ImageFolderDataset(Dataset):
    """A robust PyTorch Dataset that scans image directories and handles corrupted files gracefully."""

    def __init__(
        self,
        root_dir: Union[str, Path],
        transform: Optional[v2.Transform] = None,
        config: config = config(),
    ) -> None:
        """Initializes the dataset by scanning for images and building class maps.

        Args:
            root_dir: Path to the base folder (containing dog, cat, wild).
            transform: Optional torchvision v2 transforms.
            config: DatasetConfig instance providing target h and w parameters.

        Raises:
            ValueError: If no valid image files are located under the directory.
        """
        self.root_path = Path(root_dir)
        self.config = config

        # Supported image extensions
        self.valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

        # 1. Extract all image paths using rglob
        self.images_paths: List[Path] = [
            p for p in self.root_path.rglob("*")
            if p.is_file() and p.suffix.lower() in self.valid_extensions
        ]

        if not self.images_paths:
            raise ValueError(
                f"CRITICAL ERROR: No valid images found in '{root_dir}'. "
                f"Supported formats: {self.valid_extensions}"
            )

        # Automatically determine classes from immediate subfolders under root
        self.classes = sorted(list({p.parent.name for p in self.images_paths}))
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(self.classes)}

        # 2. Assign transform property with a fallback default pipeline
        if transform is not None:
            self.transform = transform
        else:
            # Scale -> ToTensor/Compose -> Normalize
            self.transform = v2.Compose([
                v2.Resize((self.config.h, self.config.w)),
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def __len__(self) -> int:
        """Returns the total number of identified image paths."""
        return len(self.images_paths)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Fetches, processes, and returns a specific (image, label) pair by index.

        If a non-critical error occurs (e.g. corrupted file), it logs a warning
        and returns a zero-tensor placeholder to prevent training loop failures.
        """
        img_path = self.images_paths[index]

        try:
            # Extract target label string and map to structural integer ID
            label_name = img_path.parent.name
            label_idx = self.class_to_idx[label_name]
            label_tensor = torch.tensor(label_idx, dtype=torch.long)#.unsqueeze(0)

            # Read image natively into an RGB torch tensor
            image = read_image(str(img_path), mode=ImageReadMode.RGB)

            # Apply defined or fallback transforms
            if self.transform:
                image = self.transform(image)

            return image, label_tensor

        except Exception as e:
            # Non-critical error handling: Catch file corruptions gracefully
            print(
                f"Skipping corrupted/unreadable image index {index}: '{img_path}'. "
                f"Reason: {e}. Returning zero tensor placeholder."
            )

            # Generate a clean zero-filled tensor matching expected configuration
            zero_image = torch.zeros((3, self.config.h, self.config.w), dtype=torch.float32)
            # Return -1 or a dummy label indicating an invalid target value
            dummy_label = torch.tensor(-1, dtype=torch.long)

            return zero_image, dummy_label

    def plot_images(
        self,
        paths_list: Optional[List[Union[str, Path]]] = None,
        targets_list: Optional[List[int]] = None,
        num_images: int = 4,
    ) -> None:
        """Plots a grid of images alongside their assigned folder label signatures.

        Args:
            paths_list: Optional explicit collection of file paths to show.
            targets_list: Optional explicit collection of corresponding labels.
            num_images: Maximum number of pictures to render.
        """
        fig_images: List[torch.Tensor] = []
        fig_titles: List[str] = []

        if paths_list is not None:
            # Render explicitly passed custom paths
            limit = min(len(paths_list), num_images)
            for i in range(limit):
                try:
                    p = Path(paths_list[i])
                    img = read_image(str(p), mode=ImageReadMode.RGB)
                    # Resize for plotting consistency using basic functional v2 transforms
                    img = v2.functional.resize(img, (self.config.h, self.config.w))

                    # Resolve labels if index dictionary values are provided
                    if targets_list is not None and i < len(targets_list):
                        idx = targets_list[i]
                        title = self.classes[idx] if 0 <= idx < len(self.classes) else f"Class ID: {idx}"
                    else:
                        title = p.parent.name

                    fig_images.append(img)
                    fig_titles.append(title)
                except Exception as e:
                    logger.error(f"Cannot plot custom path element: '{paths_list[i]}'. Error: {e}")
        else:
            # Default behavior: Sample directly from the dataset instance
            limit = min(len(self), num_images)
            for i in range(limit):
                img, label_tensor = self.__getitem__(np.random.randint(0, len(self)))
                label_idx = label_tensor.item()

                # Check for our dummy error placeholder value
                if label_idx == -1:
                    title = "Corrupted File Placeholder"
                else:
                    title = self.classes[label_idx]

                # Denormalize image back to 0-1 range for correct plotting exposure
                if img.max() != 0:  # Avoid dividing zero-placeholder tensors
                    img = (img - img.min()) / (img.max() - img.min())

                fig_images.append(img)
                fig_titles.append(title)

        # Execute Matplotlib Grid Plot layout routines
        if not fig_images:
            print("No valid images available to render.")
            return

        cols = min(4, len(fig_images))
        rows = (len(fig_images) + cols - 1) // cols
        fig, axs = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))

        # Flatten axes array for simple indexing if it's 2D
        if len(fig_images) == 1:
            axs = [axs]
        else:
            axs = axs.flatten() if hasattr(axs, "flatten") else axs

        for idx, (img, title) in enumerate(zip(fig_images, fig_titles)):
            # Convert channel dimension to standard shape (H, W, C) for matplotlib
            if isinstance(img, torch.Tensor):
                img_np = img.permute(1, 2, 0).cpu().numpy()
            else:
                img_np = img

            axs[idx].imshow(img_np)
            axs[idx].set_title(title)
            axs[idx].axis("off")

        # Turn off remaining empty subplots if the grid has blank cells
        for j in range(len(fig_images), len(axs)):
            axs[j].axis("off")

        plt.tight_layout()
        plt.show()

def get_loaders():
    train_ds = ImageFolderDataset(
        root_dir=Path(images_path) / "afhq" / "train",
        transform=None,
    )
    val_ds = ImageFolderDataset(
        root_dir=Path(images_path) / "afhq" / "val",
        transform=None,
    )
    # test_ds = ImageFolderDataset(
    #     root_dir=Path(images_path) / "afhq" / "test",
    #     transform=None,
    # )
    train_loader = DataLoader(train_ds, shuffle=True, **config.dataloader_configs)
    val_loader = DataLoader(val_ds, shuffle=False, **config.dataloader_configs)
    # test_loader = DataLoader(test_ds,shuffle=False, **config.dataloader_configs)
    return train_loader, val_loader

In [5]:
train_loader, val_loader = get_loaders()
images, labels = next(iter(train_loader))
print(f"Images batch shape: {images.shape}")
print(f"Labels batch shape: {labels.shape}")

Images batch shape: torch.Size([32, 3, 384, 384])
Labels batch shape: torch.Size([32])


In [6]:
for i in range(4):
    #train_loader.dataset.plot_images()
    pass

# building the model:

# Efficient Net V2 architecture

### 1. Macro-Architecture Pipeline
The network processes data sequentially through 8 stages (Stages 0 to 7):
```
[Input Image: 384 x 384 x 3]
       │
       ▼
 [Stage 0: Stem] ──► Conv3x3 (Stride 2) ──► [192 x 192 x 24]
       │
       ▼
 [Stage 1: Fused-MBConv1] ──► 2 Layers (Stride 1) ──► [192 x 192 x 24]
       │
       ▼
 [Stage 2: Fused-MBConv4] ──► 4 Layers (Stride 2) ──► [96 x 96 x 48]
       │
       ▼
 [Stage 3: Fused-MBConv4] ──► 4 Layers (Stride 2) ──► [48 x 48 x 64]
       │
       ▼
 [Stage 4: MBConv4 + SE]  ──► 6 Layers (Stride 2) ──► [24 x 24 x 128]
       │
       ▼
 [Stage 5: MBConv6 + SE]  ──► 9 Layers (Stride 1) ──► [24 x 24 x 160]
       │
       ▼
 [Stage 6: MBConv6 + SE]  ──► 15 Layers (Stride 2) ──► [12 x 12 x 256]
       │
       ▼
 [Stage 7: Head] ──► Conv1x1 + Pooling + FC ──► [Output Classes]
```

### Fused-MBConv Block (Stages 1–3)
Designed to run fast on GPU/TPU hardware by removing the depthwise convolution bottleneck.
```
           Input
             │
      ┌──────┴──────┐
      │             ▼
      │      [Conv 3x3 (Expansion + Spatial)]
      │             │
      │             ▼
      │        [BatchNorm + SiLU]
      │             │
      │             ▼
      │      [Conv 1x1 (Reduction/Project)]
      │             │
      │             ▼
      │        [BatchNorm]
      │             │
      ▼             ▼
   (Residual Connection)
             │
             ▼
          + (Add) ──► Output
```

### MBConv Block with SE (Stages 4–6)
Designed for traditional parameter savings and deep context extraction using channel attention.
```
                     Input
                       │
                ┌──────┴──────┐
                │             ▼
                │      [Conv 1x1 (Expansion)]
                │             │
                │             ▼
                │        [BatchNorm + SiLU]
                │             │
                │             ▼
                │      [Depthwise Conv 3x3]
                │             │
                │             ▼
                │        [BatchNorm + SiLU]
                │             │
                │             ▼
                │     [Squeeze-and-Excitation]
                │             │
                │             ▼
                │      [Conv 1x1 (Project)]
                │             │
                │             ▼
                │        [BatchNorm]
                │             │
                ▼             ▼
             (Residual Connection)
                       │
                       ▼
                    + (Add) ──► Output
```

In [7]:
import torch
import torch.nn as nn

# Squeeze and Extract block
class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels, reduced_channels):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, reduced_channels, 1),
            nn.SiLU(),
            nn.Conv2d(reduced_channels, in_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.se(x)

In [8]:
class FusedMBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expansion, stride, kernel_size=3):
        super().__init__()
        self.use_residual = (in_channels == out_channels) and (stride == 1)
        expanded_channels = in_channels * expansion

        # If expansion == 1, skip the first project/expand setup
        if expansion != 1:
            self.fused_layer = nn.Sequential(
                nn.Conv2d(in_channels, expanded_channels, kernel_size, stride, padding=kernel_size//2, bias=False),
                nn.BatchNorm2d(expanded_channels),
                nn.SiLU(),
                nn.Conv2d(expanded_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.fused_layer = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding=kernel_size//2, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.SiLU()
            )

    def forward(self, x):
        if self.use_residual:
            return x + self.fused_layer(x)
        return self.fused_layer(x)

In [9]:
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expansion, stride, kernel_size=3, se_ratio=0.25):
        super().__init__()
        self.use_residual = (in_channels == out_channels) and (stride == 1)
        expanded_channels = in_channels * expansion

        # 1. Expansion phase
        self.expand = nn.Sequential(
            nn.Conv2d(in_channels, expanded_channels, 1, bias=False),
            nn.BatchNorm2d(expanded_channels),
            nn.SiLU()
        ) if expansion != 1 else nn.Identity()

        # 2. Depthwise convolution
        self.depthwise = nn.Sequential(
            nn.Conv2d(expanded_channels, expanded_channels, kernel_size, stride, padding=kernel_size//2, groups=expanded_channels, bias=False),
            nn.BatchNorm2d(expanded_channels),
            nn.SiLU()
        )

        # 3. Squeeze and Excitation
        se_channels = max(1, int(in_channels * se_ratio))
        self.se = SqueezeExcitation(expanded_channels, se_channels)

        # 4. Linear Projection phase
        self.project = nn.Sequential(
            nn.Conv2d(expanded_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        res = self.expand(x)
        res = self.depthwise(res)
        res = self.se(res)
        res = self.project(res)

        if self.use_residual:
            return x + res
        return res

In [10]:
class EfficientNetV2S(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # [block_type, expansion, channels, layers, stride]
        # block_type: 0 for FusedMBConv, 1 for MBConv
        self.config = [
            [0, 1, 24, 2, 1],
            [0, 4, 48, 4, 2],
            [0, 4, 64, 4, 2],
            [1, 4, 128, 6, 2],
            [1, 6, 160, 9, 1],
            [1, 6, 256, 15, 2]
        ]

        # Stage 0: Stem input layer
        self.stem = nn.Sequential(
            nn.Conv2d(3, 24, 3, 2, 1, bias=False),
            nn.BatchNorm2d(24),
            nn.SiLU()
        )

        # Stage 1 to 6: Core Sequential Blocks
        layers = []
        in_c = 24
        for block_type, expansion, out_c, num_layers, stride in self.config:
            for i in range(num_layers):
                layer_stride = stride if i == 0 else 1
                if block_type == 0:
                    layers.append(FusedMBConv(in_c, out_c, expansion, layer_stride))
                else:
                    layers.append(MBConv(in_c, out_c, expansion, layer_stride))
                in_c = out_c
        self.stages = nn.Sequential(*layers)

        # Stage 7: Head representation layer
        self.head = nn.Sequential(
            nn.Conv2d(in_c, 1280, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(1280),
            nn.SiLU(),
        )

        # Final Classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2, inplace=True),nn.Linear(1280, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        x = self.head(x)
        x = self.classifier(x)

        return x



# Test the compilation shape output
if __name__ == "__main__":
    model = EfficientNetV2S(num_classes=10)
    test_tensor = torch.randn(1, 3, 384, 384) # Standard input size for V2-S
    output = model(test_tensor)
    print(f"Output shape: {output.shape}")  # Expected: torch.Size([1, 10])

Output shape: torch.Size([1, 10])


In [ ]:
import math
import torch
import torch.nn as nn

def initialize_efficientnet_v2(model):
    """
    Applies custom He (Kaiming) Normal initialization tailored for
    EfficientNetV2 layers using SiLU activations and Batch Normalization.
    """
    for m in model.modules():
        # 1. Initialize Convolutional Layers
        if isinstance(m, nn.Conv2d):
            # We use fan_out to preserve gradient variance during backpropagation
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)

        # 2. Initialize Batch Normalization Layers
        elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

        # 3. Initialize the Final Linear Classifier Head
        elif isinstance(m, nn.Linear):
            # Standard normal distribution scaled gently for the classifier output
            init_range = 1.0 / math.sqrt(m.out_features)
            nn.init.uniform_(m.weight, -init_range, init_range)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    # 4. Zero-initialize the last BN in each residual branch (Advanced Trick)
    # This ensures the network acts like an identity mapping at the absolute start,
    # making extremely deep networks stable right out of the gate.
    for m in model.modules():
        if hasattr(m, 'use_residual') and m.use_residual:
            # target the very last operation in the branch (the projection BatchNorm)
            if hasattr(m, 'fused_layer') and isinstance(m.fused_layer[-1], nn.BatchNorm2d):
                nn.init.zeros_(m.fused_layer[-1].weight)
            elif hasattr(m, 'project') and isinstance(m.project[-1], nn.BatchNorm2d):
                nn.init.zeros_(m.project[-1].weight)


# Modelling

In [11]:
# 1. prepare the dataloader
train_loader, val_loader = get_loaders()

# 2. prepare the criterion and model
criterion = nn.BCEWithLogitsLoss()
model = EfficientNetV2S(num_classes=3).to(config.device)
initialize_efficientnet_v2(model)

# 3. prepare the optimizer
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

In [13]:
# 3. importing the trainer
from utilities import Trainer, TrainerConfig

config=TrainerConfig()
config.task_type="multi"
trainer = Trainer(
    model=model,
    config=config,
    criterion=criterion, train_loader=train_loader,
    val_loader=val_loader
    )

In [14]:
import torch.nn as nn

# Redefine the criterion for multi-class, single-label classification
criterion = nn.CrossEntropyLoss()

# Re-initialize the trainer with the updated criterion
trainer = Trainer(
    model=model,
    config=config,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader
)

trainer.fit()

Starting pipeline on device: cuda (Task: multi)


Epoch [1/10] -> Train Loss: 0.5204 | Acc: 76.81% | F1: 0.7683 || Val Loss: 0.2332 | Acc: 91.93% | F1: 0.9197
🏆 Best validation loss updated to 0.2332. Model saved.


Epoch [2/10] -> Train Loss: 0.2021 | Acc: 92.43% | F1: 0.9235 || Val Loss: 0.1576 | Acc: 94.53% | F1: 0.9454
🏆 Best validation loss updated to 0.1576. Model saved.


Epoch [3/10] -> Train Loss: 0.1266 | Acc: 95.31% | F1: 0.9525 || Val Loss: 0.1640 | Acc: 93.87% | F1: 0.9387


KeyboardInterrupt: 